# PS-001 Story 03: Clean Workforce Data

This notebook cleans the four raw workforce CSV files, standardises the schema, exports the canonical Parquet dataset, and writes the audit and handoff artifacts required for PS-001.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import polars as pl
import yaml
from loguru import logger

workspace_root = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "shared").exists() and (path / "docs").exists()
)
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from shared.src.data_processing.workforce_cleaning import (
    clean_workforce_frame,
    combine_cleaned_frames,
)

raw_dir = workspace_root / "shared" / "data" / "1_raw" / "workforce"
parquet_path = workspace_root / "shared" / "data" / "4_processed" / "workforce_clean.parquet"
audit_path = (
    workspace_root
    / "artifacts"
    / "ps-001-workforce-data-foundation"
    / "results"
    / "tables"
    / "workforce_cleaning_audit.yml"
)
handoff_path = (
    workspace_root
    / "docs"
    / "agent-handoffs"
    / "data-cleaning"
    / "ps-001-workforce-data-foundation"
    / "cleaning_to_eda_20260422.json"
)
logger.remove()
logger.add(sys.stderr, level="INFO")

raw_paths = {
    "doctors": raw_dir / "doctors.csv",
    "nurses": raw_dir / "nurses.csv",
    "pharmacists": raw_dir / "pharmacists.csv",
    "physiotherapists": raw_dir / "physiotherapists.csv",
}
raw_paths

{'doctors': PosixPath('/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/shared/data/1_raw/workforce/doctors.csv'),
 'nurses': PosixPath('/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/shared/data/1_raw/workforce/nurses.csv'),
 'pharmacists': PosixPath('/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/shared/data/1_raw/workforce/pharmacists.csv'),
 'physiotherapists': PosixPath('/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/shared/data/1_raw/workforce/physiotherapists.csv')}

## Load Raw CSV Files

Load the four raw workforce extracts and inspect their raw shapes before standardisation.

In [2]:
raw_frames = {profession: pl.read_csv(path) for profession, path in raw_paths.items()}
{
    profession: {"shape": dataframe.shape, "columns": dataframe.columns}
    for profession, dataframe in raw_frames.items()
}

{'doctors': {'shape': (78, 4),
  'columns': ['profession', 'sector', 'year', 'headcount']},
 'nurses': {'shape': (126, 4),
  'columns': ['profession', 'sector', 'year', 'headcount']},
 'pharmacists': {'shape': (42, 4),
  'columns': ['profession', 'sector', 'year', 'headcount']},
 'physiotherapists': {'shape': (18, 4),
  'columns': ['profession', 'sector', 'year', 'headcount']}}

## Inspect Raw Data

Review the raw schema, preview rows, and sector values before cleaning.

In [3]:
raw_inspection = {}
for profession, dataframe in raw_frames.items():
    raw_inspection[profession] = {
        "schema": {column_name: str(dtype) for column_name, dtype in dataframe.schema.items()},
        "shape": dataframe.shape,
        "head": dataframe.head().to_dicts(),
        "sector_values": dataframe["sector"].unique().sort().to_list(),
    }

raw_inspection

{'doctors': {'schema': {'profession': 'String',
   'sector': 'String',
   'year': 'Int64',
   'headcount': 'Int64'},
  'shape': (78, 4),
  'head': [{'profession': 'doctors',
    'sector': 'public',
    'year': 2006,
    'headcount': 3505},
   {'profession': 'doctors',
    'sector': 'private',
    'year': 2006,
    'headcount': 2966},
   {'profession': 'doctors',
    'sector': 'not in active practice',
    'year': 2006,
    'headcount': 460},
   {'profession': 'doctors',
    'sector': 'public',
    'year': 2007,
    'headcount': 3991},
   {'profession': 'doctors',
    'sector': 'private',
    'year': 2007,
    'headcount': 3004}],
  'sector_values': ['not in active practice', 'private', 'public']},
 'nurses': {'schema': {'profession': 'String',
   'sector': 'String',
   'year': 'Int64',
   'headcount': 'Int64'},
  'shape': (126, 4),
  'head': [{'profession': 'nurses',
    'sector': 'public',
    'year': 2006,
    'headcount': 8495},
   {'profession': 'nurses',
    'sector': 'private',
 

## Run Cleaning Pipeline per Profession

Apply the shared cleaning function to each workforce file and capture the audit for each profession.

In [4]:
cleaned_frames: dict[str, pl.DataFrame] = {}
audits: list[dict[str, int | str]] = []

for profession, dataframe in raw_frames.items():
    cleaned_frame, audit = clean_workforce_frame(dataframe, profession)
    cleaned_frames[profession] = cleaned_frame.select(["year", "sector", "count", "profession"])
    audits.append(audit)
    logger.info("Audit for {}: {}", profession, audit)

audits

2026-04-22 23:32:20.014 | INFO     | __main__:<module>:8 - Audit for doctors: {'profession': 'doctors', 'rows_in': 78, 'rows_out': 78, 'nulls_dropped': 0, 'dupes_dropped': 0}
2026-04-22 23:32:20.016 | INFO     | __main__:<module>:8 - Audit for nurses: {'profession': 'nurses', 'rows_in': 126, 'rows_out': 126, 'nulls_dropped': 0, 'dupes_dropped': 0}
2026-04-22 23:32:20.018 | INFO     | __main__:<module>:8 - Audit for pharmacists: {'profession': 'pharmacists', 'rows_in': 42, 'rows_out': 42, 'nulls_dropped': 0, 'dupes_dropped': 0}
2026-04-22 23:32:20.020 | INFO     | __main__:<module>:8 - Audit for physiotherapists: {'profession': 'physiotherapists', 'rows_in': 18, 'rows_out': 18, 'nulls_dropped': 0, 'dupes_dropped': 0}


[{'profession': 'doctors',
  'rows_in': 78,
  'rows_out': 78,
  'nulls_dropped': 0,
  'dupes_dropped': 0},
 {'profession': 'nurses',
  'rows_in': 126,
  'rows_out': 126,
  'nulls_dropped': 0,
  'dupes_dropped': 0},
 {'profession': 'pharmacists',
  'rows_in': 42,
  'rows_out': 42,
  'nulls_dropped': 0,
  'dupes_dropped': 0},
 {'profession': 'physiotherapists',
  'rows_in': 18,
  'rows_out': 18,
  'nulls_dropped': 0,
  'dupes_dropped': 0}]

## Review Cleaning Audit

Summarise the row-level audit produced by the cleaning step.

In [5]:
audit_df = pl.DataFrame(audits).select([
    "profession",
    "rows_in",
    "rows_out",
    "nulls_dropped",
    "dupes_dropped",
])
audit_df

profession,rows_in,rows_out,nulls_dropped,dupes_dropped
str,i64,i64,i64,i64
"""doctors""",78,78,0,0
"""nurses""",126,126,0,0
"""pharmacists""",42,42,0,0
"""physiotherapists""",18,18,0,0


## Combine Cleaned Frames

Concatenate the profession-level cleaned frames into one canonical dataset.

In [6]:
combined = combine_cleaned_frames(cleaned_frames).select(["year", "sector", "count", "profession"])
combined.head()

year,sector,count,profession
i32,cat,i32,cat
2006,"""Public""",3505,"""doctors"""
2006,"""Private""",2966,"""doctors"""
2006,"""Not In Active Practice""",460,"""doctors"""
2007,"""Public""",3991,"""doctors"""
2007,"""Private""",3004,"""doctors"""


## Validate Final Schema & Shape

Check that the canonical dataset matches the PS-001 contract before writing outputs.

In [8]:
expected_schema = {
    "year": "Int32",
    "sector": "Categorical",
    "count": "Int32",
    "profession": "Categorical",
}
actual_schema = {column_name: str(dtype) for column_name, dtype in combined.schema.items()}
assert actual_schema == expected_schema

final_summary = {
    "shape": combined.shape,
    "schema": actual_schema,
    "describe": combined.cast({"sector": pl.String, "profession": pl.String}).describe(),
    "sector_values": combined["sector"].cast(pl.String).unique().sort().to_list(),
    "profession_values": combined["profession"].cast(pl.String).unique().sort().to_list(),
}
final_summary

{'shape': (264, 4),
 'schema': {'year': 'Int32',
  'sector': 'Categorical',
  'count': 'Int32',
  'profession': 'Categorical'},
 'describe': shape: (9, 5)
 ┌────────────┬─────────────┬────────────────────────┬─────────────┬──────────────────┐
 │ statistic  ┆ year        ┆ sector                 ┆ count       ┆ profession       │
 │ ---        ┆ ---         ┆ ---                    ┆ ---         ┆ ---              │
 │ str        ┆ f64         ┆ str                    ┆ f64         ┆ str              │
 ╞════════════╪═════════════╪════════════════════════╪═════════════╪══════════════════╡
 │ count      ┆ 264.0       ┆ 264                    ┆ 264.0       ┆ 264              │
 │ null_count ┆ 0.0         ┆ 0                      ┆ 0.0         ┆ 0                │
 │ mean       ┆ 2012.909091 ┆ null                   ┆ 2504.719697 ┆ null             │
 │ std        ┆ 3.983721    ┆ null                   ┆ 3724.627505 ┆ null             │
 │ min        ┆ 2006.0      ┆ Not In Active Practice 

## Write Parquet Output

Persist the canonical combined dataset to the shared processed data contract.

In [9]:
parquet_path.parent.mkdir(parents=True, exist_ok=True)
combined.write_parquet(parquet_path, compression="snappy")
{"parquet_path": str(parquet_path.relative_to(workspace_root)), "exists": parquet_path.exists()}

{'parquet_path': 'shared/data/4_processed/workforce_clean.parquet',
 'exists': True}

## Write Cleaning Audit YAML

Write the profession-level audit and combined schema summary to YAML.

In [10]:
audit_payload = {
    "agent": "data-cleaning",
    "ps": "ps-001-workforce-data-foundation",
    "timestamp": "2026-04-22",
    "professions": audits,
    "combined": {
        "rows_out": combined.height,
        "columns": combined.columns,
        "schema": {column_name: str(dtype) for column_name, dtype in combined.schema.items()},
    },
}
audit_path.parent.mkdir(parents=True, exist_ok=True)
with audit_path.open("w", encoding="utf-8") as file_handle:
    yaml.safe_dump(audit_payload, file_handle, sort_keys=False, allow_unicode=False)

{"audit_path": str(audit_path.relative_to(workspace_root)), "audit_rows": audit_df.to_dicts()}

{'audit_path': 'artifacts/ps-001-workforce-data-foundation/results/tables/workforce_cleaning_audit.yml',
 'audit_rows': [{'profession': 'doctors',
   'rows_in': 78,
   'rows_out': 78,
   'nulls_dropped': 0,
   'dupes_dropped': 0},
  {'profession': 'nurses',
   'rows_in': 126,
   'rows_out': 126,
   'nulls_dropped': 0,
   'dupes_dropped': 0},
  {'profession': 'pharmacists',
   'rows_in': 42,
   'rows_out': 42,
   'nulls_dropped': 0,
   'dupes_dropped': 0},
  {'profession': 'physiotherapists',
   'rows_in': 18,
   'rows_out': 18,
   'nulls_dropped': 0,
   'dupes_dropped': 0}]}

## Write Handoff JSON

Persist the required handoff document for the exploratory-analysis agent.

In [11]:
handoff_payload = {
    "agent": "data-cleaning",
    "ps": "ps-001-workforce-data-foundation",
    "status": "success",
    "timestamp": "2026-04-22",
    "outputs": [
        "shared/src/data_processing/workforce_cleaning.py",
        "shared/tests/unit/test_workforce_cleaning.py",
        "artifacts/ps-001-workforce-data-foundation/scripts/clean_workforce_data.py",
        "artifacts/ps-001-workforce-data-foundation/notebooks/03_clean_workforce_data.ipynb",
        "shared/data/4_processed/workforce_clean.parquet",
        "artifacts/ps-001-workforce-data-foundation/results/tables/workforce_cleaning_audit.yml",
    ],
    "clean_parquet_path": "shared/data/4_processed/workforce_clean.parquet",
    "clean_parquet_schema": {
        "year": "Int32",
        "sector": "Categorical",
        "count": "Int32",
        "profession": "Categorical",
    },
    "unit_tests": "pass",
    "next_agent": "exploratory-analysis",
}
handoff_path.parent.mkdir(parents=True, exist_ok=True)
handoff_path.write_text(json.dumps(handoff_payload, indent=2), encoding="utf-8")

{
    "handoff_path": str(handoff_path.relative_to(workspace_root)),
    "final_shape": combined.shape,
    "final_schema": handoff_payload["clean_parquet_schema"],
}

{'handoff_path': 'docs/agent-handoffs/data-cleaning/ps-001-workforce-data-foundation/cleaning_to_eda_20260422.json',
 'final_shape': (264, 4),
 'final_schema': {'year': 'Int32',
  'sector': 'Categorical',
  'count': 'Int32',
  'profession': 'Categorical'}}